# 🧹 Car Data — EDA + Data Cleaning + Model Training (GUIDE)

Yeh notebook ek **step-by-step guide** hai, solution nahi. Har section me:
- Kya karna hai (instructions)
- Kya check/socho (hints)
- Ek khaali code cell jahan tumhe khud code likhna hai

Dataset: `messy_car_data.csv` — isme jaan-bujh kar gandagi dali gayi hai:
missing values, duplicate rows, inconsistent text casing/spacing, units mixed
in numeric columns (`$`, `km`, `L`), placeholder junk (`N/A`, `unknown`, `-`),
outliers, aur ek junk `Unnamed: 0` column.

**Goal:** Isko saaf karke Linear/Ridge/Lasso Regression model train karna
(jaise pichle notebook me tha), lekin ab poora cleaning process khud karna hai.


## Step 0 — Imports

**Kya karna hai:** Zaroori libraries import karo — `pandas`, `numpy`,
`matplotlib`/`seaborn` (visualization ke liye), aur `sklearn` se
`LabelEncoder`, `train_test_split`, `LinearRegression`, `Ridge`, `Lasso`,
`r2_score`.


In [ ]:
import pandas as pd
import numpy as np


: 

## Step 1 — Data Load & First Look

**Kya karna hai:**
1. `messy_car_data.csv` ko load karo.
2. `.shape`, `.head()`, `.tail()`, `.info()`, `.describe(include='all')` se
   dataset ka overview lo.

**Socho:**
- Kitni rows/columns hain?
- Kaunse columns numeric hone chahiye the lekin `object`/string dikh rahe hain?
- Koi column bekaar (irrelevant) to nahi lag raha?


In [ ]:
# TODO: load the dataset and explore it
df = pd.read_csv("messy_car_data.csv")


print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nDataset Info:")
df.info()

print("\nSummary Statistics:")
display(df.describe(include='all'))


## Step 2 — Missing Values ka Audit

**Kya karna hai:**
1. Har column me kitne missing values (`NaN`) hain — `df.isna().sum()`.
2. Percentage me bhi dekho (`df.isna().mean()*100`).
3. Woh rows dhundo jo **completely empty** hain (saare columns `NaN`).

**Socho:**
- Kaunse columns me missing % itna zyada hai ki drop karna better hoga?
- Kaunse columns me impute (mean/median/mode) karna sahi rahega?


In [ ]:
# TODO: missing value audit
# Missing count aur percentage
missing_count = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing_count, 'Missing %': missing_percent})
print(missing_df[missing_df['Missing Count'] > 0])

# Completely empty rows check
empty_rows = df.isna().all(axis=1).sum()
print(f"\nCompletely empty rows: {empty_rows}")



## Step 3 — Junk / Placeholder Values Dhundo

**Kya karna hai:**
Kuch missing values `NaN` ki jagah junk text ki tarah chhupe hain — jaise
`"N/A"`, `"unknown"`, `"-"`, `"??"`, `"null"`, `"None"` (string, NaN nahi).

1. Har categorical column (`Brand`, `Body`, `Engine Type`, `Registration`)
   ke unique values print karo (`df['col'].unique()`).
2. Jo bhi placeholder-type junk values milein, unhe asli `np.nan` me convert
   karo (`.replace()`).

**Socho:** Agar yeh step skip kiya to aage LabelEncoder in junk values ko
bhi ek alag "category" samjhega — galat hoga.


In [ ]:
# TODO: find and replace placeholder junk values with real NaN
cat_cols = ['Brand', 'Body', 'Engine Type', 'Registration']

print("Unique values before cleaning placeholders:")
for col in cat_cols:
    print(f"\n{col}: {df[col].unique()}")

# Common placeholder junk values ko asli np.nan me convert karo
junk_placeholders = ['unknown', 'UNKNOWN', 'Unknown', '-', '??', 'N/A', 'null', 'None', '?']
df[cat_cols] = df[cat_cols].replace(junk_placeholders, np.nan)



## Step 4 — Duplicate Rows Hatao

**Kya karna hai:**
1. `df.duplicated().sum()` se dekho kitni exact duplicate rows hain.
   (Tip: `Unnamed: 0` / index-type column ko ignore karke check karo,
   warna woh unique bana dega duplicates ko.)
2. `df.drop_duplicates()` se hatao.
3. Bekaar `Unnamed: 0` jaisa index column bhi drop kar do.

**Socho:** Duplicate rows model ko ek hi pattern baar-baar sikha kar bias
kar sakti hain.


In [ ]:
# TODO: remove duplicate rows and drop the junk index column
# Unnamed: 0 ko ignore karke duplicates count karo
cols_without_id = [c for c in df.columns if c != 'Unnamed: 0']
print("Duplicate rows count:", df.duplicated(subset=cols_without_id).sum())

# Duplicates remove karo aur junk Unnamed: 0 column drop karo
df = df.drop_duplicates(subset=cols_without_id).reset_index(drop=True)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print("Shape after removing duplicates:", df.shape)



## Step 5 — Text Columns Standardize Karo

**Kya karna hai:**
Categorical columns (`Brand`, `Body`, `Engine Type`, `Registration`) me
values ka casing aur spacing inconsistent hai — jaise `"BMW"`, `"bmw"`,
`"  BMW  "` teeno ek hi cheez hain lekin computer inhe alag samjhega.

1. Har text column ko `.str.strip()` (extra spaces hatao) aur consistent
   case (jaise `.str.title()` ya `.str.upper()`) me convert karo.
2. Dubara `.unique()` check karo ki ab clean categories hain.

**Socho:** `Registration` column ko sirf `Yes`/`No` (2 categories) me
consistent karna hai.


In [ ]:
# TODO: strip whitespace + fix inconsistent casing in categorical columns
# Whitespace strip aur casing fix
for col in ['Brand', 'Body', 'Engine Type']:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace({'Nan': np.nan})

# Registration column ko Yes / No me standardize karo
df['Registration'] = df['Registration'].astype(str).str.strip().str.title()
df['Registration'] = df['Registration'].replace({'Nan': np.nan})

print("Categories after standardizing:")
for col in cat_cols:
    print(f"{col}: {df[col].dropna().unique()}")



## Step 6 — Numeric Columns ke Types Fix Karo

**Kya karna hai:**
`Price`, `Mileage`, `EngineV` columns me kuch values units/symbols ke saath
string ban gayi hain:
- `Price`: kuch jagah `"$12,500.00"` jaisi string hai
- `Mileage`: kuch jagah `"150,000 km"` jaisi string hai
- `EngineV`: kuch jagah `"2.0L"` jaisi string hai

1. In columns se `$`, `,`, `km`, `L` jaise characters hatao
   (`.str.replace()` regex ke saath useful hoga).
2. Phir `pd.to_numeric(..., errors='coerce')` se numeric type me convert
   karo (jo convert nahi ho payega woh automatically `NaN` ban jayega).
3. `df.dtypes` se confirm karo ab yeh columns numeric hain.

**Socho:** `errors='coerce'` kyun use kar rahe hain?


In [ ]:
# TODO: clean and convert Price, Mileage, EngineV to proper numeric dtype
# Price, Mileage, EngineV se units aur symbols ($ , km L) hatao
for col in ['Price', 'Mileage', 'EngineV']:
    df[col] = df[col].astype(str).str.replace(r'[\$,kmL\s]', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df[['Price', 'Mileage', 'EngineV', 'Year']].dtypes)
display(df[['Price', 'Mileage', 'EngineV', 'Year']].head())



## Step 7 — Outliers / Invalid Values Handle Karo

**Kya karna hai:** Dataset me kuch clearly ghalat (invalid) values hain:
- `Price <= 0`
- `EngineV > 10` (real cars ka engine volume itna nahi hota)
- `Mileage < 0`
- `Year` bahut purana (jaise `< 1900`) ya future me (jaise `> 2026`)

1. Boolean filters bana kar dekho kitni rows in conditions me aati hain.
2. Decide karo: **drop** karoge ya **cap/clip** karoge in values ko.
   (Pichle "clean" notebook me `EngineV <= 10` filter use hua tha — waisa
   hi soch is baar bhi baaki columns ke liye.)

**Socho:** Har outlier row ko turant drop karna hamesha sahi nahi hota —
pehle samjho kya sahi lag raha hai is data ke liye.


In [ ]:
# TODO: detect and handle outliers/invalid values
# Invalid / impossible values filter karo:
# 1. Price > 0
# 2. EngineV <= 10 (99.9 jaise impossible volumes drop honge)
# 3. Mileage >= 0
# 4. 1900 <= Year <= 2026
df = df[df['Price'] > 0]
df = df[(df['EngineV'].isna()) | (df['EngineV'] <= 10)]
df = df[(df['Mileage'].isna()) | (df['Mileage'] >= 0)]
df = df[(df['Year'].isna()) | ((df['Year'] >= 1900) & (df['Year'] <= 2026))]

print("Shape after filtering outliers/invalid records:", df.shape)



## Step 8 — Missing Values ko Handle Karo (Impute ya Drop)

Ab jab text aur numeric columns clean ho chuke hain, missing values
(`NaN`) ko dubara audit karo (Step 2 wale numbers ab badal chuke honge).

**Kya karna hai:**
1. Jin rows me target column (`Price`) hi missing hai, unhe drop karo
   (bina price ke training possible nahi).
2. Baaki numeric columns (`Mileage`, `EngineV`, `Year`) ke missing values
   ko median se impute karo (ya jo tumhe sahi lage).
3. Categorical columns (`Brand`, `Body`, `Engine Type`, `Registration`) ke
   missing values ko mode (sabse common value) se ya `"Unknown"` category
   se fill karo.

**Socho:** Mean vs Median — outliers ke saath kaunsa better hota hai?


In [ ]:
# TODO: handle remaining missing values

# Target variable 'Price' jahan missing ho usko drop karo
df = df.dropna(subset=['Price'])

# Numeric columns ko Median se impute karo
for col in ['Mileage', 'EngineV', 'Year']:
    df[col] = df[col].fillna(df[col].median())

# Categorical columns ko Mode (most frequent) se impute karo
for col in ['Brand', 'Body', 'Engine Type', 'Registration']:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after imputation:")
print(df.isna().sum())


## Step 9 — Quick Visual EDA (Optional but recommended)

**Kya karna hai:** `matplotlib`/`seaborn` se:
1. `Price` ki distribution plot karo (histogram) — kya woh skewed hai?
2. `Price` vs `Mileage`, `Price` vs `Year` ka scatter plot dekho.
3. `Brand` ke hisaab se average `Price` ka bar chart banao.

**Socho:** Agar `Price` right-skewed hai, to kya `log(Price)` use karna
better hoga model ke liye? (Pichle "clean" notebook me `Log_price` isi
wajah se banaya gaya tha.)


In [ ]:
# TODO: (optional) visualize distributions and relationships
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
sns.histplot(df['Price'], kde=True, bins=30, color='royalblue')
plt.title("Price Distribution (Right-Skewed)")

plt.subplot(1, 3, 2)
sns.scatterplot(x='Mileage', y='Price', data=df, alpha=0.6)
plt.title("Price vs Mileage")

plt.subplot(1, 3, 3)
sns.scatterplot(x='Year', y='Price', data=df, alpha=0.6)
plt.title("Price vs Year")

plt.tight_layout()
plt.show()



## Step 10 — Feature Engineering + Encoding

**Kya karna hai:**
1. `Log_price = np.log(Price)` column banao (target variable).
2. `Model` column drop karo (bahut zyada unique values, useful nahi lagta).
3. Categorical columns (`Brand`, `Body`, `Engine Type`, `Registration`) ko
   `LabelEncoder` se numeric me convert karo — **encoders ko ek dictionary
   me save karo** (future me app/deployment ke liye zaroori hoga).
4. `Notes` jaisa mostly-empty irrelevant column bhi drop kar do.

**Socho:** Encoders ko save karna kyun zaroori hai (pichle deployment guide
me yeh cheez discuss hui thi)?


In [ ]:
# # TODO: feature engineering + label encoding (save encoders in a dict!)

# # Skewness remove karne ke liye target ka log transform
# df['Log_price'] = np.log(df['Price'])

# # Irrelevant columns drop karo
# drop_cols = [c for c in ['Price', 'Model', 'Notes'] if c in df.columns]
# df = df.drop(columns=drop_cols)

# # Categorical columns ko encode karo aur dictionary me save karo
# encoders = {}
# for col in ['Brand', 'Body', 'Engine Type', 'Registration']:
#     le = LabelEncoder()
#     df[col] = le.fit_transform(df[col])
#     encoders[col] = le

# print("Encoded DataFrame preview:")
# display(df.head())


## Step 11 — Train/Test Split

**Kya karna hai:**
1. `X` = saare feature columns (Log_price aur Price chhod kar).
2. `y` = `Log_price`.
3. `train_test_split(X, y, test_size=0.2, random_state=42)` use karo.


In [ ]:
# # TODO: train/test split
# X = df.drop(columns=['Log_price'])
# y = df['Log_price']

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")




## Step 12 — Model Training (Linear, Ridge, Lasso)

**Kya karna hai:**
1. Teeno models train karo: `LinearRegression()`, `Ridge(alpha=...)`,
   `Lasso(alpha=...)`.
2. Har ek ka `r2_score` test set par nikalo.
3. Compare karo — kaunsa best perform kar raha hai?

**Socho:** Ridge/Lasso me `alpha` badalne se performance kaise change hoti
hai? Ek-do different alpha values try karo.


In [ ]:
# TODO: train Linear, Ridge, Lasso and compare R2 scores




## Step 13 — Final Recap

Apne aap se yeh questions poochho (short notes likh lo yahan markdown me):

- Cleaning se pehle vs baad me kitni rows reh gayi?
- Sabse zyada missing values kis column me the, aur tumne unhe kaise handle kiya?
- Kaunsa model best raha aur kyun?
- Agar yeh model deploy karna ho (Streamlit), to kya-kya save karna zaroori hoga?

*(Yahan apne answers likho)*


In [ ]:
# (Optional) final summary code / notes
